# Testing outliners
Testing diffrent methods of outliners detection and chceking if adding a new coloum is_outlier and droppiing them, to see if it improves model accuarcy

## IQR

In [1]:
import pandas as pd 
from scipy.stats import zscore
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from model_evaluation import EvalConfig, EvalModel

df = pd.read_csv("Data/features_30_sec.csv")

In [2]:
drop_cols = ['filename', 'length', 'label']
data = df.drop(columns=drop_cols, errors='ignore')

In [3]:
def iqr_mask(data, m=1.5):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 -Q1
    outliers_mask_iqr = ((data < (Q1 - m * IQR)) | (data > (Q3 + m * IQR))).any(axis=1)
    return outliers_mask_iqr

## Z- score

In [4]:
def z_score_mask(df, m=3):
    z_matrix = zscore(df)
    abs_z_matrix = np.abs(z_matrix)
    z_outliers_mask = (abs_z_matrix > m).any(axis=1)
    
    return z_outliers_mask


## Isolation Forest

In [5]:
def iso_mask(df, contamination=0.02):
    iso_model = IsolationForest(contamination=contamination, random_state=42)
    predictions = iso_model.fit_predict(df)
    mask = (predictions == -1)
    return mask


## Deleting outlniers

In [6]:
from model_evaluation import EvalConfig, EvalModel
from sklearn.pipeline import Pipeline
import os
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

input_csv = "Data/features_30_sec.csv" 
drop_cols = ['filename', 'length', 'label'] 
target_col = 'label'

pipelines = [
    Pipeline([('rf', RandomForestClassifier(random_state=42))]),
    Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(C=10, gamma="scale")),
]),
    Pipeline([('scaler', StandardScaler()),('lr', LogisticRegression(max_iter=1000))]),
    Pipeline([('xgb', XGBClassifier(eval_metric='logloss'))])
            ]

experiments = [
    {"name": "Baseline (No filtering)", "func": None, "params": {}},
    
    {"name": "IQR (m=3.0)", "func": iqr_mask, "params": {"m": 3.0}},
    {"name": "IQR (m=2.5)", "func": iqr_mask, "params": {"m": 2.5}},
    
    {"name": "Z-Score (m=3)", "func": z_score_mask, "params": {"m": 3}},
    {"name": "Z-Score (m=2.5)", "func": z_score_mask, "params": {"m": 2.5}},

    {"name": "IsolationForest (25%)", "func": iso_mask, "params": {"contamination": 0.25}},
    {"name": "IsolationForest (30%)", "func": iso_mask, "params": {"contamination": 0.3}},
]

In [7]:
temp_csv_name = "temp_csv_data.csv"

In [8]:
df_full = pd.read_csv(input_csv)

In [12]:
results =[]
from sklearn.preprocessing import LabelEncoder
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

for exp in experiments:
    le = LabelEncoder()
    df_full[target_col] = le.fit_transform(df_full[target_col])
    print("Mapping od labels:", dict(zip(le.classes_, le.transform(le.classes_))))
    
    for pipeline in pipelines:
        curr_df = df_full.copy()
        model_name = pipeline.steps[-1][1].__class__.__name__
    
        if exp["func"] is not None:
            X = curr_df.drop(columns = drop_cols, errors='ignore')
            mask = exp["func"](X, **exp["params"])
    
            curr_df = curr_df[~mask]
        removed_count = len(df_full) - len(curr_df)
        curr_df.to_csv(temp_csv_name, index=False)
    
        config = EvalConfig(
            train_path=temp_csv_name,
            pipeline=pipeline,
            drop_cols=drop_cols,
            target_col=target_col
        )
    
        eval_model = EvalModel(config)
        res = eval_model.evaluate()
        acc = res['report']['accuracy']
    
        results.append({
            "Model": model_name,
            "Method": exp["name"],
            "Removed Rows": removed_count,
            "Accuracy": acc,
        })
if os.path.exists(temp_csv_name):
    os.remove(temp_csv_name)

Mapping od labels: {np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5): np.int64(5), np.int64(6): np.int64(6), np.int64(7): np.int64(7), np.int64(8): np.int64(8), np.int64(9): np.int64(9)}
Mapping od labels: {np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5): np.int64(5), np.int64(6): np.int64(6), np.int64(7): np.int64(7), np.int64(8): np.int64(8), np.int64(9): np.int64(9)}
Mapping od labels: {np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5): np.int64(5), np.int64(6): np.int64(6), np.int64(7): np.int64(7), np.int64(8): np.int64(8), np.int64(9): np.int64(9)}
Mapping od labels: {np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3), np.int64(4): np.int64(4), np.int64(5

In [13]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False)

results_df

,Model,Method,Removed Rows,Accuracy
13,SVC,Z-Score (m=3),264,0.769231
5,SVC,IQR (m=3.0),328,0.767327
2,LogisticRegression,Baseline (No filtering),0,0.766667
1,SVC,Baseline (No filtering),0,0.750000
21,SVC,IsolationForest (25%),250,0.746667
9,SVC,IQR (m=2.5),388,0.739130
25,SVC,IsolationForest (30%),300,0.738095
4,RandomForestClassifier,IQR (m=3.0),328,0.722772
3,XGBClassifier,Baseline (No filtering),0,0.720000
17,SVC,Z-Score (m=2.5),402,0.716667
